# Complexity-03e — Barrières d'exposants et word RAM

**Navigation** : [Index](README.md) | [Version Python](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb) | [03 — Hiérarchie temporelle](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03f — Réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb)

**Objectifs** : lire des exposants rationnels, exécuter les neuf instructions d'un word RAM et confronter un témoin au décompte manuel ; mesurer un contrôle 3SUM quadratique. **Prérequis** : C#, complexité et complément à deux. **Durée estimée** : 60 minutes.

Ce carnet lit une [formalisation publiée](https://github.com/anthropics/formal-math/tree/main/3sum-apsp), sans exécuter les programmes des auteurs ni `lake build`. Les cinq claims sont des existences de programmes et leurs constantes ne sont pas optimisées. Le contrôle quadratique ne mesure pas le gain annoncé.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Diagnostics;
using System.Globalization;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
const int SEED=20261028;
Console.WriteLine($"{System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BigInteger ; graine {SEED}");

.NET 10.0.14 ; BigInteger ; graine 20261028


## Cinq claims cités

| Énoncé | Problème | Borne citée |
|---|---|---|
| Theorem 19 | Exact Triangle | $O(n^{2,9983})$ |
| Theorem 22 | 3SUM | $O(n^{1,9992})$ |
| Theorem 22 | produit (min,+) | $O(n^{2,99942})$ |
| Theorem 22 | APSP | $O(n^{2,99942})$ |
| Corollary 39 | k-Clique de poids nul | $O(n^{k-0,0017\lfloor k/3\rfloor})$ |

Le fichier `EndStatement.lean` énonce ces propositions. Le papier donne deux bornes pour le théorème 22 : les claims reprennent les plus petites. Le BigO rationnel élève les deux membres à la puissance du dénominateur.

Pour Theorem 22, le papier présente deux algorithmes : les bornes numériques citées sont 1,99923 et 1,9992 pour 3SUM, puis 2,99949 et 2,99942 pour le produit (min,+) et APSP. Les claims formels prennent les seconds nombres, plus petits. Il s'agit de bornes théoriques citées, pas de chronos exécutés ici.

In [2]:
foreach(var (numerateur,denominateur) in new[]{(2499,1250),(149971,50000)})
 Console.WriteLine($"r = {numerateur}/{denominateur} : T(n)^{denominateur} <= K * n^{numerateur}, n >= 2");

r = 2499/1250 : T(n)^1250 <= K * n^2499, n >= 2


r = 149971/50000 : T(n)^50000 <= K * n^149971, n >= 2


Les deux fractions sont exactes, et non des arrondis introduits ici. Le projet décrit les preuves mais ce carnet ne les recharge pas.

### Exercice 1 — quotient de puissances

Retourner $n^{\Delta}$ pour un exposant positif. Tester deux couples non triviaux ; une fois la fonction complétée, calculer les écarts des deux algorithmes pour 3SUM et pour (min,+)/APSP, puis leurs facteurs à $10^6$ et $10^{12}$.

In [3]:
double? Facteur(double n,double delta) { // TODO : calculer n puissance delta
 return null;
}
var facteur=Facteur(16,0.5);
if(facteur is null) Console.WriteLine("Exercice 1 à compléter : quotient des puissances.");
else { var autre=Facteur(1000,1.0/3); if(!double.IsFinite(facteur.Value)||Math.Abs(facteur.Value-4)>1e-9||autre is null||!double.IsFinite(autre.Value)||Math.Abs(autre.Value-10)>1e-9) throw new Exception("Facteur incorrect"); Console.WriteLine("Deux fixtures du facteur validées."); foreach(var (probleme,delta) in new[]{("3SUM",0.00003),("min-plus",0.00007),("APSP",0.00007)}) { var f6=Facteur(1e6,delta); var f12=Facteur(1e12,delta); if(f6 is null || f12 is null || !double.IsFinite(f6.Value) || !double.IsFinite(f12.Value) || Math.Abs(f6.Value-Math.Pow(1e6,delta))>1e-9 || Math.Abs(f12.Value-Math.Pow(1e12,delta))>1e-9) throw new Exception("Facteurs incorrects"); Console.WriteLine($"{probleme} : écart {delta:F5}, facteur 10^6={f6:F6}, 10^12={f12:F6}"); } }

Exercice 1 à compléter : quotient des puissances.


## Machine à neuf instructions

`one`, `add`, `sub`, `mul`, `load`, `store`, `bltz`, `accept`, `reject` coûtent un pas chacun, arrêt compris. La mémoire, initialement nulle hors entrée, est indexée par des entiers signés ; les mots se calculent modulo $2^W$. `load` et `store` interprètent l'adresse indirecte en complément à deux. Une position hors programme rejette. Il n'y a ni division, ni décalage, ni opération bit à bit, et la seule constante est 1.

La machine ci-dessous est un interpréteur de lecture et non une preuve des claims. Le budget borne son exécution.

In [4]:
record Instruction(string Op, int A=0, int B=0, int C=0);
record Resultat(bool Verdict, Dictionary<BigInteger,BigInteger> Memoire, int Pas);
BigInteger NonSigne(BigInteger v,int w) { var m=BigInteger.One<<w; return ((v%m)+m)%m; }
BigInteger Signe(BigInteger v,int w) { v=NonSigne(v,w); var m=BigInteger.One<<w; return v >= (m>>1) ? v-m : v; }
Dictionary<BigInteger,BigInteger> Charger(int w,params int[] mots) { var d=new Dictionary<BigInteger,BigInteger>(); for(int i=0;i<mots.Length;i++) d[i]=NonSigne(mots[i],w); return d; }
Resultat Exec(Instruction[] p,Dictionary<BigInteger,BigInteger> mem0,int w,int budget) {
 var mem=new Dictionary<BigInteger,BigInteger>(mem0); int pc=0; BigInteger Lire(BigInteger a)=>mem.GetValueOrDefault(a,BigInteger.Zero);
 for(int pas=1;pas<=budget;pas++) { var ins=pc>=0 && pc<p.Length ? p[pc] : new Instruction("reject");
  switch(ins.Op) {
   case "one": mem[ins.A]=NonSigne(1,w); pc++; break;
   case "add": mem[ins.A]=NonSigne(Lire(ins.B)+Lire(ins.C),w); pc++; break;
   case "sub": mem[ins.A]=NonSigne(Lire(ins.B)-Lire(ins.C),w); pc++; break;
   case "mul": mem[ins.A]=NonSigne(Lire(ins.B)*Lire(ins.C),w); pc++; break;
   case "load": mem[ins.A]=Lire(Signe(Lire(ins.B),w)); pc++; break;
   case "store": mem[Signe(Lire(ins.A),w)]=Lire(ins.B); pc++; break;
   case "bltz": pc=Signe(Lire(ins.A),w)<0 ? ins.B : pc+1; break;
   case "accept": return new(true,mem,pas);
   case "reject": return new(false,mem,pas);
   default: throw new ArgumentException("Instruction inconnue");
  }
 } return null;
}
Console.WriteLine("Interpréteur chargé : neuf instructions ; budget et arrêt comptés.");

Interpréteur chargé : neuf instructions ; budget et arrêt comptés.


### Débordement et adressage

Tester explicitement la réduction modulo et l'adresse indirecte négative, en dehors du témoin 3SUM où les sommes restent dans la plage signée. Le débordement ne doit pas être confondu avec la somme entière.

In [5]:
var programmeTest=new[]{new Instruction("add",3,1,2),new Instruction("store",4,3),new Instruction("load",5,4),new Instruction("mul",6,1,2),new Instruction("accept")};
var memTest=Charger(4,0,7,2,0,-1); var test=Exec(programmeTest,memTest,4,10);
if(test==null || Signe(test.Memoire[3],4)!=-7 || test.Memoire[-1]!=9 || test.Memoire[5]!=9 || test.Memoire[6]!=14 || test.Pas!=5) throw new Exception("Erreur débordement/adresse");
Console.WriteLine($"W=4 : 7+2 devient {Signe(test.Memoire[3],4)}, case -1={test.Memoire[-1]}, 7*2 modulo 16={test.Memoire[6]}, {test.Pas} pas.");

W=4 : 7+2 devient -7, case -1=9, 7*2 modulo 16=14, 5 pas.


### Exercice 2 — fabriquer une constante

La machine ne possède que `one`. Construire un programme qui laisse $k$ en cellule 1, par doubles et additions. Un `accept` termine le programme ; les fixtures contrôlent 10 et 13, pas une constante unique.

In [6]:
Instruction[] ConstruireConstante(int k) { // TODO : fabriquer k, puis accept
 return null;
}
var construit=ConstruireConstante(10);
if(construit is null) Console.WriteLine("Exercice 2 à compléter : programme de construction.");
else { var a=Exec(construit,new(),16,100); var autre=ConstruireConstante(13); var b=autre is null ? null : Exec(autre,new(),16,100); if(a is null || !a.Verdict || a.Memoire.GetValueOrDefault(1)!=10 || b is null || !b.Verdict || b.Memoire.GetValueOrDefault(1)!=13) throw new Exception("Programme incorrect"); Console.WriteLine("Constantes 10 et 13 validées."); }

Exercice 2 à compléter : programme de construction.


## Témoin fermé à trois nombres

Le programme additionne les trois mots, fabrique zéro, puis rejette si la somme ou son opposé est négatif. Les chemins attendus sont 7 pas pour zéro, 6 pour une somme négative et 7 pour une somme positive. Ce test suppose l'absence de débordement signé ; la batterie utilise de petits entiers.

In [7]:
Instruction[] P3={new("add",10,1,2),new("add",10,10,3),new("sub",11,10,10),new("sub",12,11,10),new("bltz",10,7),new("bltz",12,7),new("accept"),new("reject")};
foreach(var (a,b,c,attendu,pas) in new[]{(-5,2,3,true,7),(-3,-2,-1,false,6),(1,2,3,false,7)}) { var r=Exec(P3,Charger(16,3,a,b,c),16,100); if(r is null||r.Verdict!=attendu||r.Pas!=pas) throw new Exception("Témoin incorrect"); Console.WriteLine($"[{a},{b},{c}] : {r.Verdict}, {r.Pas} pas (attendu {attendu}, {pas})"); }
var rng=new Random(SEED); int accords=0;
for(int i=0;i<300;i++) { int[] x=Enumerable.Range(0,3).Select(_=>rng.Next(-20,21)).ToArray(); if(rng.NextDouble()<0.35) { int a=rng.Next(3),b=(a+1+rng.Next(2))%3; x[3-a-b]=-x[a]-x[b]; } var r=Exec(P3,Charger(16,3,x[0],x[1],x[2]),16,100); if(r!=null && r.Verdict==(x.Sum()==0)) accords++; }
Console.WriteLine($"Batterie : {accords}/300 accords, System.Random distinct de Python.");
var debord=Exec(P3,Charger(4,3,7,7,2),4,100); if(debord is null || !debord.Verdict) throw new Exception("Débordement attendu");
var hors=Exec(Array.Empty<Instruction>(),new(),4,1); var borne=Exec(P3,Charger(16,3,0,0,0),16,1); if(hors is null || hors.Verdict || hors.Pas!=1 || borne is not null) throw new Exception("Budget ou rejet hors programme incorrect");
Console.WriteLine("W=4 : 7+7+2 vaut 0 modulo 16 : accept malgré somme entière 16 ; hors-programme rejette en un pas et budget épuisé ne rend aucun verdict.");

[-5,2,3] : True, 7 pas (attendu True, 7)


[-3,-2,-1] : False, 6 pas (attendu False, 6)


[1,2,3] : False, 7 pas (attendu False, 7)


Batterie : 300/300 accords, System.Random distinct de Python.


W=4 : 7+7+2 vaut 0 modulo 16 : accept malgré somme entière 16 ; hors-programme rejette en un pas et budget épuisé ne rend aucun verdict.


Le témoin vérifie trois chemins et une batterie finie, sans établir la correction générale de la machine. Les graines sont fixes mais les générateurs des deux langages diffèrent. À W=4, une somme entière de 16 devient zéro modulo 16 : ce contre-exemple est hors du domaine sans débordement du témoin borné.

## Contrôle 3SUM quadratique

Tri et deux pointeurs par ancre, sur des entrées positives sans solution : la boucle parcourt son travail complet. Trois répétitions par taille, médiane, puis pente log-log. Le chronomètre et les constantes dépendent du runtime.

In [8]:
bool TroisSomme(int[] xs) { var t=xs.OrderBy(x=>x).ToArray(); for(int i=0;i<t.Length-2;i++) { int g=i+1,d=t.Length-1; long cible=-(long)t[i]; while(g<d) { long s=(long)t[g]+t[d]; if(s==cible)return true; if(s<cible)g++; else d--; } } return false; }
if(!TroisSomme(new[]{-3,1,2}) || TroisSomme(new[]{1,2,3})) throw new Exception("Fixtures 3SUM incorrectes");
var tailles=new[]{1000,2000,4000,8000}; var mesures=new List<(int n,double t)>(); rng=new Random(SEED); TroisSomme(new[]{1,2,3});
foreach(int n in tailles) { int[] x=Enumerable.Range(0,n).Select(_=>rng.Next(1,n*n+1)).ToArray(); var temps=new List<double>(); bool rep=true; for(int j=0;j<3;j++) { var sw=Stopwatch.StartNew(); rep=TroisSomme(x); sw.Stop(); temps.Add(sw.Elapsed.TotalSeconds); } temps.Sort(); mesures.Add((n,temps[1])); Console.WriteLine($"n={n}: {temps[1]*1000:F2} ms, médiane de 3, réponse {rep}"); }
var lx=mesures.Select(t=>Math.Log(t.n)).ToArray(); var ly=mesures.Select(t=>Math.Log(t.t)).ToArray(); double mx=lx.Average(),my=ly.Average(); double pente=lx.Zip(ly,(x,y)=>(x-mx)*(y-my)).Sum()/lx.Sum(x=>(x-mx)*(x-mx));
Console.WriteLine($"Pente log-log : {pente:F3} ; quotient des puissances à 8000 : {Math.Pow(8000,0.0008):F4}");

n=1000: 1.49 ms, médiane de 3, réponse False


n=2000: 5.81 ms, médiane de 3, réponse False


n=4000: 21.66 ms, médiane de 3, réponse False


n=8000: 88.26 ms, médiane de 3, réponse False


Pente log-log : 1.956 ; quotient des puissances à 8000 : 1.0072


Le quotient $8000^{0,0008}$ compare les seules puissances. Il ne prédit pas le rapport des temps : les constantes et termes cachés sont inconnus ; aucune implémentation de la borne citée n'est exécutée ici.

### Exercice 3 — comptage des comparaisons

Sur une entrée strictement positive triée, compter les itérations de la boucle à deux pointeurs pour toutes les ancres : c'est un sous-calcul du coût. Puis proposer une traduction explicite d'une itération en instructions `load`, `sub`, `bltz` et déplacements d'indices, et dériver un ordre de grandeur en pas de `exec` pour le décideur tri + deux pointeurs. Justifier aussi le coût du tri et la taille des mots. Comparer aux chronos sans assimiler des itérations à des pas machine : leur nombre dépend de la traduction choisie. Dans cette exécution, la médiane passe de 1,49 ms à 88,26 ms entre 1 000 et 8 000 éléments, pour une pente mesurée de 1,956 ; ces temps varient selon la machine et ne mesurent que le contrôle quadratique.

In [9]:
long? CompterComparaisons(int n) { // TODO : sommer le nombre d'itérations sur n éléments positifs
 return null;
}
var compte=CompterComparaisons(4);
if(compte is null) Console.WriteLine("Exercice 3 à compléter : compter les itérations.");
else { var autre=CompterComparaisons(7); if(compte!=3 || autre!=15) throw new Exception("Comptage incorrect"); Console.WriteLine("Fixtures 4 et 7 validées."); }

Exercice 3 à compléter : compter les itérations.


## Ce que la source ne prouve pas

Le README exclut notamment les bornes pour entrées réelles et la section 5.1 ; les constantes ne sont pas optimisées. Les hypothèses de difficulté ne sont pas définies dans la formalisation ; les claims énoncent des existences. Le README ne rapporte pas d'exécution de Comparator. La machine est plus faible qu'un word RAM standard par ses opérations limitées, mais autorise une multiplication en un pas et une mémoire adressable par mot. Le programme est fixé avec un nombre $b$ et doit tenir pour toute taille de mot $W \ge b(1+\lfloor\log_2 n\rfloor)$. Quatre gratuités décrites dans MACHINE.md sont : mémoire hors entrée initialisée à zéro, cellules nommées dans le programme hors de portée des mots, choix de la pente $b$ après $\kappa$, et entrées de phases gratuites ; le document précise qu'elles ne changent pas l'exposant.

La série situe cette formalisation hors Mathlib face au manque d'une couche `TIME(f)` discuté dans [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb). [03d](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) examine une autre annonce et [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) replace ces travaux dans le panorama. Le fichier d'énoncé n'importe rien ; cela ne remplace pas une vérification locale des preuves.

## À retenir

Les cinq bornes sont citées, le témoin et le contrôle quadratique sont mesurés. Trois exercices répartis explorent le quotient des exposants, la construction d'une constante et un comptage d'itérations.

## Vérifiez votre compréhension

1. Pourquoi 7 et 6 pas ? 2. Le quotient des puissances prédit-il les temps ? 3. Pourquoi relever à la puissance 1250 ?

<details><summary>Réponses</summary>

1. Le premier chemin atteint `accept` et le second saute plus tôt à `reject`, arrêt inclus. 2. Non : constantes et termes cachés restent inconnus. 3. $1,9992=2499/1250$, ce qui permet d'exprimer la borne avec des puissances entières.
</details>

## Pour aller plus loin

- [EndStatement.lean](https://github.com/anthropics/formal-math/blob/main/3sum-apsp/EndStatement.lean), [README et MACHINE.md](https://github.com/anthropics/formal-math/tree/main/3sum-apsp).
- [03f — Réductions](Complexity-03f-ReductionsTriangle-Lopsided-Python.ipynb) et [07c — Bilan](Complexity-07c-LeBilanMathocalypse-Python.ipynb).

**Navigation** : [Index](README.md) | [Version Python](Complexity-03e-Barrieres-Exposants-WordRAM-Python.ipynb)